# Rule-Testing Framework — demo

A quick, runnable walkthrough of `rule_eval`: given a **baseline** matcher, a **candidate**
rule, and a **labeled** eval set, produce a before/after comparison with Bayesian credible
intervals and a **ship / reject / needs-more-data** verdict.

Everything a "rule" needs to be is a function `features -> bool`, so a threshold change, a
scoring rule, or a blocking rule all plug in identically. See `DESIGN.md` for the full spec.

> In Databricks: put `rule_eval.py` next to this notebook (or `%pip install` it as a wheel)
> and remove the `sys.path` line below.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../evaluation"))   # so `import rule_eval` finds ../evaluation/rule_eval.py
import numpy as np
import rule_eval as re

print("rule_eval loaded")

## 1. A labeled gold-standard set (synthetic)

Each `LabeledPair` has `features` (what the rule reads), a gold `is_true_match` label, and
`strata` for stratified sampling. We deliberately include a **`dob_present: "no"`** stratum
— the CMS gap — so we can read how a rule degrades on missing-DOB records.

In [ ]:
def make_pairs(seed=7):
    rng = np.random.default_rng(seed)
    strata = {"age_band": ["0-17", "18-64", "65+"],
              "name_commonality": ["common", "rare"],
              "dob_present": ["yes", "no"]}
    def s():
        return {k: str(rng.choice(v)) for k, v in strata.items()}
    pairs = []
    for i in range(2000):  # true matches
        pairs.append(re.LabeledPair({"score": float(rng.uniform(0.3, 1.0)), "exact_id": False},
                                    is_true_match=True, strata=s(), pair_id=f"T{i}"))
    for i in range(120):   # true matches the baseline misses, but an exact-id signal catches
        pairs[i] = re.LabeledPair({"score": float(rng.uniform(0.30, 0.49)), "exact_id": True},
                                  is_true_match=True, strata=s(), pair_id=f"Tgain{i}")
    for i in range(2000):  # non-matches (hard negatives near the boundary)
        pairs.append(re.LabeledPair({"score": float(rng.uniform(0.0, 0.55)), "exact_id": False},
                                    is_true_match=False, strata=s(), pair_id=f"N{i}"))
    rng.shuffle(pairs)
    return pairs

pairs = make_pairs()
dev, holdout = re.stratified_split(pairs, holdout_frac=0.70, strata_keys=["dob_present"], seed=1)
print(f"dev={len(dev)}  holdout(protected eval)={len(holdout)}  base_rate={np.mean([p.is_true_match for p in holdout]):.3f}")

## 2. Define rules as `features -> bool`

`baseline` = score ≥ 0.5. Two candidates:
- `lenient` = lower the threshold to 0.40 (a **threshold change**)
- `trust_exact_id` = baseline OR an exact-id signal (a **new scoring rule**)

In [ ]:
baseline       = lambda f: f["score"] >= 0.50
lenient        = lambda f: f["score"] >= 0.40
trust_exact_id = lambda f: (f["score"] >= 0.50) or bool(f.get("exact_id"))

## 3. Before/after — candidate that lowers the threshold
More recall, but it also lets in more false positives → **REJECT** on the safety rule.

In [ ]:
rep = re.compare(baseline, lenient, holdout, baseline_name="score>=0.5", candidate_name="score>=0.4")
print(re.format_report(rep))

## 4. Before/after — candidate that adds an exact-id signal
Gains true matches with no new false positives → **SHIP**. Note the paired churn
(gained vs lost) and the credible-interval plot.

In [ ]:
rep2 = re.compare(baseline, trust_exact_id, holdout,
                  baseline_name="score>=0.5", candidate_name="trust-exact-id")
print(re.format_report(rep2))

In [ ]:
# In Databricks: display(re.report_to_dataframe(rep2))
df = re.report_to_dataframe(rep2)
print(df.to_string(index=False))
fig = re.plot_credible_intervals(rep2)   # in Databricks this renders inline
fig

## 5. Read the CMS gap slice (missing DOB)
Always check the `dob_present: "no"` stratum — a rule can win overall yet degrade where
DOB is absent (CMS-sourced records).

In [ ]:
missing_dob = [p for p in holdout if p.strata["dob_present"] == "no"]
print(f"missing-DOB holdout pairs: {len(missing_dob)}")
rep_slice = re.compare(baseline, trust_exact_id, missing_dob, candidate_name="trust-exact-id (missing-DOB slice)")
print(re.format_report(rep_slice))

## 6. How many labels do we need?
`min_sample_size` gives the labels/arm to detect an effect; `detectable_delta` inverts it
for the count you actually have.

In [ ]:
for d in (0.02, 0.03, 0.05):
    print(f"detect a {d:.0%} shift near a 0.90 rate -> {re.min_sample_size(0.90, d)} labels/arm")
print(f"with 1500 labels/arm near 0.90, smallest detectable shift = {re.detectable_delta(1500, 0.90):.3f}")

# cheap robustness check: metric variance across 5 folds of the holdout
print(re.kfold_metric_variance(baseline, holdout, metric="tpr", k=5))

## Verdict policy (safety-first)
- **REJECT** if FPR or Precision decisively regresses (`P(better) ≤ 0.05`) — a wrong-patient
  release risk is disqualifying.
- **SHIP** if no safety regression and the primary metric decisively improves (`P(better) ≥ 0.95`).
- **NEEDS MORE DATA** otherwise (credible intervals still overlap) — get more labels or lean on
  the paired test.

Run this on the **protected holdout only**, and re-run after any change to matching logic,
normalization, or thresholds.